# Découpage spatial des données OSM — Région Casablanca-Settat

## Projet SIG : Évaluation de la vulnérabilité aux inondations urbaines dans le Grand Casablanca

Ce notebook présente le découpage spatial des données géographiques
utilisées dans le cadre du projet, à partir de la limite administrative
de la région Casablanca-Settat.

### Objectifs
- Définir l'emprise géographique de la zone d'étude.
- Découper les couches géographiques OpenStreetMap (OSM).
- Enregistrer les résultats dans une structure de dossiers organisée.


### Sources de données
- **Source 1 :** OpenStreetMap — données géographiques générales.

### Référentiel géographique
- **Zone d'étude :** Casablanca-Settat, Maroc.
- **CRS de la limite utilisée  / source:** EPSG:4326.
- **Format de sortie :** GeoPackage (`.gpkg`).
- 
Les données brutes sont conservées intactes. Les couches découpées
sont enregistrées dans le dossier `donnees_decoupes/`.


## 1. Définition de l'emprise et découpage des données OSM

Cette étape consiste à identifier la limite administrative de
Casablanca-Settat, à vérifier sa géométrie et à l'utiliser pour
découper les couches OpenStreetMap.

Les couches obtenues sont enregistrées au format GeoPackage dans
`donnees_decoupes/casablanca_settat/osm/`.

In [1]:

from pathlib import Path
import geopandas as gpd
import pandas as pd

# -----------------------------------------------------------------------------
# CONFIGURATION DES CHEMINS
# -----------------------------------------------------------------------------
# Le notebook est exécuté depuis Geodata/scripts
ROOT = Path.cwd().resolve().parent

# Données brutes OSM
OSM_DIR = ROOT / "donnees_brutes" / "osm"
ADMIN_PATH = OSM_DIR / "gis_osm_adminareas_a_free_1.shp"

# Données traitées : à l'extérieur de scripts
OUT_DIR = (
    ROOT / "donnees_decoupes"
    / "casablanca_settat"
    / "osm"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Dossier racine :", ROOT)
print("Dossier OSM brut :", OSM_DIR.resolve())
print("Dossier OSM traité :", OUT_DIR.resolve())


# 1. Lire les limites administratives
admin = gpd.read_file(ADMIN_PATH)

# 2. Sélectionner la région Casablanca-Settat
limite = admin[
    (admin["osm_id"].astype(str) == "2424085")
    & (admin["fclass"] == "admin_level4")
].copy()

if limite.empty:
    raise ValueError(
        "Limite Casablanca-Settat introuvable : vérifie osm_id et fclass."
    )

if limite.crs is None:
    raise ValueError("Le CRS de la limite administrative est absent.")

# Réparer la géométrie si nécessaire
limite["geometry"] = limite.geometry.make_valid()
limite = limite[
    limite.geometry.notna() & ~limite.geometry.is_empty
]
limite = limite.dissolve()

# Sauvegarder la limite utilisée
limite.to_file(
    OUT_DIR.parent / "limite_casablanca_settat.gpkg",
    driver="GPKG"
)

print("Limite administrative chargée.")
print("CRS de la limite :", limite.crs)
print("Dossier de sortie :", OUT_DIR)

# 3. Découper chaque shapefile OSM
shapefiles = [
    p for p in OSM_DIR.glob("*.shp")
    if p.name.lower() != ADMIN_PATH.name.lower()
]

rapport = []

for fichier in sorted(shapefiles):
    print(f"\nTraitement : {fichier.name}")

    try:
        couche = gpd.read_file(fichier)

        if couche.empty:
            print("  -> Couche vide, ignorée.")
            rapport.append({
                "couche": fichier.name,
                "statut": "vide",
                "entites": 0
            })
            continue

        if couche.crs is None:
            print("  -> CRS absent, couche ignorée.")
            rapport.append({
                "couche": fichier.name,
                "statut": "CRS absent",
                "entites": 0
            })
            continue

        # Mettre la couche dans le CRS de la limite
        if couche.crs != limite.crs:
            couche = couche.to_crs(limite.crs)

        # Retirer les géométries nulles ou vides
        couche = couche[
            couche.geometry.notna() & ~couche.geometry.is_empty
        ].copy()

        if couche.empty:
            print("  -> Aucune géométrie exploitable.")
            rapport.append({
                "couche": fichier.name,
                "statut": "sans géométrie",
                "entites": 0
            })
            continue

  # Découpage spatial exact par la limite régionale
        decoupee = gpd.clip(couche, limite)

        if decoupee.empty:
            print("  -> Aucune entité dans la région.")
            rapport.append({
                "couche": fichier.name,
                "statut": "aucune entité dans la région",
                "entites": 0
            })
            continue

        # Export GeoPackage : évite les limites du format Shapefile
        sortie = OUT_DIR / f"{fichier.stem}_casablanca_settat.gpkg"
        decoupee.to_file(sortie, driver="GPKG")

        print(f"  -> {len(decoupee)} entités enregistrées")

        rapport.append({
            "couche": fichier.name,
            "statut": "OK",
            "entites": len(decoupee),
            "fichier_sortie": sortie.name
        })

    except Exception as e:
        print(f"  -> ERREUR : {e}")
        rapport.append({
            "couche": fichier.name,
            "statut": f"Erreur : {e}",
            "entites": 0
        })

# 4. Afficher le bilan
bilan = pd.DataFrame(rapport)

print("\n" + "=" * 65)
print("BILAN DU DÉCOUPAGE OSM")
print("=" * 65)
display(bilan)

print("\nNombre de couches traitées :", len(bilan))
print("Couches exportées :", (bilan["statut"] == "OK").sum())
print("Dossier de sortie :", OUT_DIR)


Dossier racine : C:\Users\espacegamers.ma\OneDrive\Documents\geoai-inondation-casablanca
Dossier OSM brut : C:\Users\espacegamers.ma\OneDrive\Documents\geoai-inondation-casablanca\donnees_brutes\osm
Dossier OSM traité : C:\Users\espacegamers.ma\OneDrive\Documents\geoai-inondation-casablanca\donnees_decoupes\casablanca_settat\osm
Limite administrative chargée.
CRS de la limite : EPSG:4326
Dossier de sortie : C:\Users\espacegamers.ma\OneDrive\Documents\geoai-inondation-casablanca\donnees_decoupes\casablanca_settat\osm

Traitement : gis_osm_buildings_a_free_1.shp


C:\Users\espacegamers.ma\AppData\Local\Temp\ipykernel_19124\3905716849.py:100: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  couche.geometry.notna() & ~couche.geometry.is_empty


  -> 85205 entités enregistrées

Traitement : gis_osm_landuse_a_free_1.shp


C:\Users\espacegamers.ma\AppData\Local\Temp\ipykernel_19124\3905716849.py:100: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  couche.geometry.notna() & ~couche.geometry.is_empty


  -> 11826 entités enregistrées

Traitement : gis_osm_natural_a_free_1.shp
  -> 44 entités enregistrées

Traitement : gis_osm_natural_free_1.shp
  -> 18864 entités enregistrées

Traitement : gis_osm_places_a_free_1.shp
  -> 14 entités enregistrées

Traitement : gis_osm_places_free_1.shp
  -> 404 entités enregistrées

Traitement : gis_osm_pofw_a_free_1.shp
  -> 408 entités enregistrées

Traitement : gis_osm_pofw_free_1.shp
  -> 93 entités enregistrées

Traitement : gis_osm_pois_a_free_1.shp


C:\Users\espacegamers.ma\AppData\Local\Temp\ipykernel_19124\3905716849.py:100: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  couche.geometry.notna() & ~couche.geometry.is_empty


  -> 3570 entités enregistrées

Traitement : gis_osm_pois_free_1.shp
  -> 7456 entités enregistrées

Traitement : gis_osm_protected_areas_a_free_1.shp
  -> Aucune entité dans la région.

Traitement : gis_osm_railways_free_1.shp
  -> 1084 entités enregistrées

Traitement : gis_osm_roads_free_1.shp
  -> 143099 entités enregistrées

Traitement : gis_osm_traffic_a_free_1.shp
  -> 1076 entités enregistrées

Traitement : gis_osm_traffic_free_1.shp
  -> 9543 entités enregistrées

Traitement : gis_osm_transport_a_free_1.shp
  -> 65 entités enregistrées

Traitement : gis_osm_transport_free_1.shp
  -> 2693 entités enregistrées

Traitement : gis_osm_water_a_free_1.shp
  -> 544 entités enregistrées

Traitement : gis_osm_waterways_free_1.shp
  -> 506 entités enregistrées

BILAN DU DÉCOUPAGE OSM


,couche,statut,entites,fichier_sortie
0,gis_osm_buildings_a_free_1.shp,OK,85205,gis_osm_buildings_a_free_1_casablanca_settat.gpkg
1,gis_osm_landuse_a_free_1.shp,OK,11826,gis_osm_landuse_a_free_1_casablanca_settat.gpkg
2,gis_osm_natural_a_free_1.shp,OK,44,gis_osm_natural_a_free_1_casablanca_settat.gpkg
3,gis_osm_natural_free_1.shp,OK,18864,gis_osm_natural_free_1_casablanca_settat.gpkg
4,gis_osm_places_a_free_1.shp,OK,14,gis_osm_places_a_free_1_casablanca_settat.gpkg
5,gis_osm_places_free_1.shp,OK,404,gis_osm_places_free_1_casablanca_settat.gpkg
6,gis_osm_pofw_a_free_1.shp,OK,408,gis_osm_pofw_a_free_1_casablanca_settat.gpkg
7,gis_osm_pofw_free_1.shp,OK,93,gis_osm_pofw_free_1_casablanca_settat.gpkg
8,gis_osm_pois_a_free_1.shp,OK,3570,gis_osm_pois_a_free_1_casablanca_settat.gpkg
9,gis_osm_pois_free_1.shp,OK,7456,gis_osm_pois_free_1_casablanca_settat.gpkg



Nombre de couches traitées : 19
Couches exportées : 18
Dossier de sortie : C:\Users\espacegamers.ma\OneDrive\Documents\geoai-inondation-casablanca\donnees_decoupes\casablanca_settat\osm


In [2]:
# -----------------------------------------------------------------------------
# COUCHE COMMUNES — région Casablanca-Settat découpée en communes
#   PARTIE A : à AJOUTER À LA SUITE de ton bloc de découpage OSM
#              (utilise les variables : ROOT, admin, limite)
#   
# -----------------------------------------------------------------------------

# =============================================================================
# PARTIE A — EXTRACTION DES COMMUNES
# =============================================================================
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
from shapely.geometry import MultiPolygon

CRS_CIBLE = "EPSG:26191"

CLEAN_DIR = ROOT / "donnees_traitees" / "casablanca_settat_clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
COMMUNES_PATH = CLEAN_DIR / "communes_casablanca_settat_clean.gpkg"

N_COMMUNES_HCP = 153        # HCP, rapport régional 2024 (région Casablanca-Settat)
FCLASS_COMMUNE = None       # ex. "admin_level6" ; None = choix automatique (voir diagnostic)


def union_geom(gs):
    return gs.union_all() if hasattr(gs, "union_all") else gs.unary_union


# A1. Préparer toutes les entités administratives
adm = admin.copy()
adm = adm[adm.geometry.notna() & ~adm.geometry.is_empty].copy()
adm["geometry"] = adm.geometry.make_valid()
if adm.crs != limite.crs:
    adm = adm.to_crs(limite.crs)

# A2. Garder celles qui sont DANS la région (point représentatif dans la limite)
region_geom = union_geom(limite.geometry)
pts = adm.geometry.representative_point()
dans = adm[pts.within(region_geom)].copy()

# A3. Diagnostic : quel niveau administratif correspond aux communes ?
diag = dans["fclass"].value_counts().sort_index()
print("Entités administratives dans la région, par niveau :")
print(diag.to_string())
print(f"\nRéférence HCP : {N_COMMUNES_HCP} communes")

if FCLASS_COMMUNE is None:
    candidats = diag.drop("admin_level4", errors="ignore")
    if candidats.empty:
        raise ValueError("Aucun niveau administratif inférieur à la région dans le shapefile.")
    FCLASS_COMMUNE = (candidats - N_COMMUNES_HCP).abs().idxmin()
    print(f"\nNiveau retenu automatiquement : {FCLASS_COMMUNE} "
          f"({diag[FCLASS_COMMUNE]} entités). Vérifie qu'il correspond bien aux communes.")

nb = int(diag.get(FCLASS_COMMUNE, 0))
if abs(nb - N_COMMUNES_HCP) > 0.1 * N_COMMUNES_HCP:
    print(f"[ATTENTION] {nb} entités contre {N_COMMUNES_HCP} attendues "
          f"(écart > 10 %). Fixe FCLASS_COMMUNE à la main ou vérifie les niveaux.")

# A4. Construire la couche communes
communes = dans[dans["fclass"] == FCLASS_COMMUNE][["osm_id", "name", "geometry"]].copy()
communes = communes.rename(columns={"name": "nom"})

communes["osm_id"] = pd.to_numeric(communes["osm_id"], errors="coerce")
communes = communes.dropna(subset=["osm_id"])
communes["osm_id"] = communes["osm_id"].astype("int64")

n_avant = len(communes)
communes = communes[communes.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
if len(communes) < n_avant:
    print(f"[INFO] {n_avant - len(communes)} géométries non polygonales écartées.")

communes["geometry"] = communes.geometry.apply(
    lambda g: MultiPolygon([g]) if g.geom_type == "Polygon" else g
)
communes = communes.to_crs(CRS_CIBLE)
communes = communes.sort_values("nom").reset_index(drop=True)
communes["id"] = np.arange(1, len(communes) + 1)
communes["surface_km2"] = (communes.area / 1e6).round(2)
communes = communes[["id", "osm_id", "nom", "surface_km2", "geometry"]]

# A5. Contrôles
region_proj = limite.to_crs(CRS_CIBLE)
region_geom_proj = union_geom(region_proj.geometry)
union_c = union_geom(communes.geometry)

recouvrement = union_c.intersection(region_geom_proj).area / region_geom_proj.area
chevauchement = communes.area.sum() / union_c.area

print(f"\nCommunes extraites         : {len(communes)}")
print(f"Couverture de la région    : {recouvrement:.1%}")
print(f"Somme surfaces / union     : {chevauchement:.3f}  (> 1 = chevauchements, ex. arrondissements)")
print(f"Géométries invalides       : {int((~communes.is_valid).sum())}")
print(f"Noms manquants             : {int(communes['nom'].isna().sum())}")

# A6. Sauvegarde GeoPackage (prêt pour l'import PostGIS)
if COMMUNES_PATH.exists():
    COMMUNES_PATH.unlink()
communes.to_file(COMMUNES_PATH, layer="commune", driver="GPKG")
print(f"\n[OK] Couche communes : {COMMUNES_PATH.resolve()}")



Entités administratives dans la région, par niveau :
fclass
admin_level10     18
admin_level11      7
admin_level4       1
admin_level5       9
admin_level6      49
admin_level7      55
admin_level8     153
admin_level9      18

Référence HCP : 153 communes

Niveau retenu automatiquement : admin_level8 (153 entités). Vérifie qu'il correspond bien aux communes.

Communes extraites         : 153
Couverture de la région    : 100.0%
Somme surfaces / union     : 1.000  (> 1 = chevauchements, ex. arrondissements)
Géométries invalides       : 0
Noms manquants             : 0

[OK] Couche communes : C:\Users\espacegamers.ma\OneDrive\Documents\geoai-inondation-casablanca\donnees_traitees\casablanca_settat_clean\communes_casablanca_settat_clean.gpkg
